In [1]:
from datetime import datetime
from langchain.tools import tool



# first tool
@tool
def get_current_time():
    """Returns the current system time."""

    return datetime.now().strftime("%H:%M:%S")

/Users/aman/enterprise-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# second tool
@tool
def calculator(a: float, b: float):
    """
    Adds two numbers.
    """
    return a + b

In [3]:
from langchain.chat_models import init_chat_model

def get_model():
    return init_chat_model(model="groq:llama-3.3-70b-versatile")

model = get_model()

In [4]:
tool_registry = {
    "get_current_time":get_current_time,
    "calculator":calculator
}



In [5]:
model_with_tools = model.bind_tools(list(tool_registry.values()))

In [32]:
response1 = model_with_tools.invoke("what time it is?")
# print(response)
response1.tool_calls

[{'name': 'get_current_time',
  'args': {},
  'id': '7s2wy54rs',
  'type': 'tool_call'}]

In [22]:
response = model_with_tools.invoke(
    "What is 45 plus 89?"
)
tool_call = response.tool_calls[0]
tool_name = tool_call["name"]
tool_args = tool_call["args"]

tool = tool_registry[tool_name]



tool_result = tool.invoke(tool_args)

tool_result



134.0

In [23]:
from langchain_core.messages import ToolMessage

tool_message = ToolMessage(
    content=str(tool_result),
    tool_call_id=tool_call["id"]
)
print(tool_message)

content='134.0' tool_call_id='jhv4h0a3t'


In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
messages = [
   SystemMessage(content="You are a helpful AI Assistant"),
   HumanMessage(content="what time it is and what is 45 plus 89?")
]

while True:
   response = model_with_tools.invoke(messages)
   print("\nAI RESPONSE:")
   print(response)
   messages.append(response)

   if not response.tool_calls:
      break
   for tool_call in response.tool_calls:
      tool_name = tool_call["name"]
      tool_args = tool_call["args"]

      print("\nTOOL:", tool_name)
      print("ARGS:", tool_args)

      # find tool
      tool_function = tool_registry[tool_name]

      # execute tool
      try:
         tool_result = tool_function.invoke(tool_args)
      except Exception as e:
         tool_result = f"Tool execution failed: {str(e)}"
      
      print("RESULT:", tool_result)

      # send response back to LLM
      tool_message = ToolMessage(content=str(tool_result),tool_call_id=tool_call["id"])

      messages.append(tool_message)

print(response.content)


AI RESPONSE:
content='' additional_kwargs={'tool_calls': [{'id': '480qhqt3r', 'function': {'arguments': '{}', 'name': 'get_current_time'}, 'type': 'function'}, {'id': 'r756xdhrr', 'function': {'arguments': '{"a":45,"b":89}', 'name': 'calculator'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 273, 'total_tokens': 300, 'completion_time': 0.074950559, 'completion_tokens_details': None, 'prompt_time': 0.014515417, 'prompt_tokens_details': None, 'queue_time': 0.056388833, 'total_time': 0.089465976}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fe524-916d-7ab2-b1a2-35888bdef6fa-0' tool_calls=[{'name': 'get_current_time', 'args': {}, 'id': '480qhqt3r', 'type': 'tool_call'}, {'name': 'calculator', 'args': {'a': 45, 'b': 89}, 'id': 'r756xdhrr', 'type': 'tool_call'}] invalid_tool_calls=[] u